**Silver as a stream, with a watermark**

In [0]:
from pyspark.sql.functions import window, col, sum as _sum, count

silver_stream = spark.readStream.table("silver_events")

gold_stream = (
    silver_stream
    .withWatermark("event_time", "10 minutes")
    .filter(col("event_type") == "PURCHASE")
    .groupBy(window(col("event_time"), "5 minutes"))
    .agg(
        _sum("price").alias("total_revenue"),
        count("*").alias("num_purchases")
    )
)

**OutputMode changes here**

In [0]:
gold_query = (
    gold_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/demo_series1/checkpoints/gold_revenue/")
    .outputMode("complete")   # not "append" — aggregates get updated as new data arrives
    .trigger(availableNow=True)
    .table("gold_revenue_by_window")
)

**Verify**

In [0]:

spark.sql("SELECT * FROM gold_revenue_by_window ORDER BY window DESC").display()

window,total_revenue,num_purchases
"List(2026-08-03T17:55:00.000Z, 2026-08-03T18:00:00.000Z)",158217.66,221
"List(2026-08-03T17:00:00.000Z, 2026-08-03T17:05:00.000Z)",144293.98999999996,190
"List(2026-08-03T16:50:00.000Z, 2026-08-03T16:55:00.000Z)",127641.14999999998,182
